In [1]:
import pandas as pd

/Users/maryamzakiyya/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [17]:
df = pd.read_csv('all_files.csv')
df

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
2,2,11-13-20 Kaur and Singh.pdf,"Comments Received After September 30, 2020",1,0.000008,"November 13th, 2020\n\nDear Superintendent Thu..."
3,3,11-19-20 Lamont.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sami Lamont Sent: Wednesday, November 18..."
4,4,11-18-20 Jensen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sine Hwang Jensen Sent: Tuesday, Novembe..."
...,...,...,...,...,...,...
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [4]:
import pandas as pd

# Example DataFrame

# Check which rows are empty or just spaces
empty_mask = df['text'].str.strip().eq('') | df['text'].isna()

df[empty_mask]


,Unnamed: 0,file_name,folder_name,submissions,weight,text


In [6]:
import string
def is_illegible(s):
    if pd.isna(s) or s.strip() == '':
        return False  # treat empty as separate case
    return any(c not in string.printable for c in s)


illegible_rows = df[df['text'].apply(is_illegible)]

illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
5,5,11-17-20 Yardeni.pdf,"Comments Received After September 30, 2020",1,0.000008,----------------------------------------------...
13,13,11-13-20 Epstein et al.pdf,"Comments Received After September 30, 2020",1,0.000008,Dear Members of the Instructional Quality Comm...
15,15,11-13-2- Khalili.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Zoha Khalili Sent: Friday, November 13, ..."
16,16,11-11-20 DePass.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Linval DePass Sent: Wednesday, November ..."
17,17,11-13-20 Homsi.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Pauline Homsi Sent: Friday, November 13,..."
...,...,...,...,...,...,...
7709,7710,10-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 2, 20..."
7710,7711,10-2-20 Boyd.pdf,"Comments Received After September 30, 2020",1,0.000008,"October 2, 2020\nPresident\nE. Toby Boyd\n\nTO..."
7711,7712,10-14-20 George.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: IQC\nSent: Wednesday, October 14, 2020 1..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."


In [19]:
import re
import pandas as pd

def legibility_metrics(s):
    if pd.isna(s) or not isinstance(s, str) or s.strip() == '':
        return {"alpha_ratio": 0, "word_ratio": 0, "weird_char_ratio": 0}
    
    s_clean = s.replace('\n', ' ').replace('\r', ' ')
    total_chars = len(s_clean)
    alpha_chars = sum(c.isalpha() for c in s_clean)
    alpha_ratio = alpha_chars / total_chars if total_chars > 0 else 0

    words = re.findall(r'[A-Za-z]{2,}', s_clean)
    word_ratio = len(words) / (total_chars / 5) if total_chars > 0 else 0

    weird_char_ratio = sum(not (c.isalnum() or c.isspace()) for c in s_clean) / total_chars

    return {
        "alpha_ratio": alpha_ratio,
        "word_ratio": word_ratio,
        "weird_char_ratio": weird_char_ratio
    }

# Apply and add to df
metrics = df['text'].apply(legibility_metrics).apply(pd.Series)
df_metrics = pd.concat([df, metrics], axis=1)


In [20]:
df_metrics[['alpha_ratio', 'word_ratio', 'weird_char_ratio']].describe()

,alpha_ratio,word_ratio,weird_char_ratio
count,7717.000000,7717.000000,7717.000000
mean,0.790383,0.738118,0.030825
std,0.025821,0.044306,0.011637
min,0.333333,0.043838,0.000000
25%,0.783047,0.709681,0.024486
50%,0.795695,0.736152,0.028606
75%,0.805656,0.764563,0.033719
max,0.871165,0.948827,0.253937


In [22]:
alpha_low = df_metrics['alpha_ratio'].mean() - 3 * df_metrics['alpha_ratio'].std()   # unusually low alphabetic %
word_low = df_metrics['word_ratio'].mean() - 3 * df_metrics['word_ratio'].std()     # unusually low English word %
weird_high = df_metrics['weird_char_ratio'].mean() + 3 * df_metrics['weird_char_ratio'].std()  # unusually high symbol %

illegible_rows = df[
    (df_metrics['alpha_ratio'] < alpha_low) |
    (df_metrics['word_ratio'] < word_low) |
    (df_metrics['weird_char_ratio'] > weird_high)
]

illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
5,5,11-17-20 Yardeni.pdf,"Comments Received After September 30, 2020",1,0.000008,----------------------------------------------...
13,13,11-13-20 Epstein et al.pdf,"Comments Received After September 30, 2020",1,0.000008,Dear Members of the Instructional Quality Comm...
86,86,11-11-20 Guttenberg.pdf,"Comments Received After September 30, 2020",1,0.000008,-----Original Message----From: martaguttenberg...
133,133,11-13-20 Silton.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: On Behalf Of Lynn Silton Sent: Friday, N..."
139,139,11-10-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Tuesday, November 10, ..."
...,...,...,...,...,...,...
7673,7674,10-27-20 Cohen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ken Cohen\nSent: Monday, October 26, 202..."
7684,7685,11-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, November 2, 2..."
7691,7692,10-14-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 13, ..."
7692,7693,10-26-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."


In [26]:
# Recalculate new (looser) thresholds
alpha_low_2 = df_metrics['alpha_ratio'].mean() - 2 * df_metrics['alpha_ratio'].std()
word_low_2 = df_metrics['word_ratio'].mean() - 2 * df_metrics['word_ratio'].std()
weird_high_2 = df_metrics['weird_char_ratio'].mean() + 2 * df_metrics['weird_char_ratio'].std()

# Run again to find more rows that look illegible under looser rules
illegible_rows_looser = df[
    (df_metrics['alpha_ratio'] < alpha_low_2) |
    (df_metrics['word_ratio'] < word_low_2) |
    (df_metrics['weird_char_ratio'] > weird_high_2)
]

# Keep only new ones not in the first batch
new_illegible_rows = illegible_rows_looser.loc[
    ~illegible_rows_looser.index.isin(illegible_rows.index)
]

new_illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
105,105,11-12-20 Kaufman.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Jon Kaufman Sent: Thursday, November 12,..."
175,175,11-18-20 Hsiao.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Rod Hsiao Sent: Wednesday, November 18, ..."
177,177,11-13-20 Baha.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Omar Baha Sent: Friday, November 13, 202..."
200,200,11-13-20 Webber Attachment 1.pdf,"Comments Received After September 30, 2020",1,0.000008,Arabic 367: American Identity in the World Ara...
250,250,11-13-20 Chang Attachment 2.pdf,"Comments Received After September 30, 2020",1,0.000008,1 This lesson was submitted by a member(s) of ...
...,...,...,...,...,...,...
7699,7700,10-13-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 13, ..."
7700,7701,10-1-20 Klein.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Steve Klein\nSent: Thursday, October 1, ..."
7705,7706,10-19-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 19, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."


In [28]:
# attempt to catch Kane Gold's
def strong_illegibility_check(s):
    if pd.isna(s) or not isinstance(s, str) or s.strip() == '':
        return True  # treat empty or missing as illegible

    total_chars = len(s)
    alpha_ratio = sum(c.isalpha() for c in s) / total_chars
    words = re.findall(r'[A-Za-z]{2,}', s)
    word_ratio = len(words) / (total_chars / 5 if total_chars else 1)
    weird_char_ratio = sum(not (c.isalnum() or c.isspace()) for c in s) / total_chars
    
    short_words = [w for w in words if len(w) < 3]
    short_word_ratio = len(short_words) / (len(words) if words else 1)
    
    long_weird_seq = bool(re.search(r'[^A-Za-z0-9\s]{5,}', s))

    # Flag as illegible if it fails any of these stricter criteria
    return (word_ratio < 0.3) or (short_word_ratio > 0.5) or long_weird_seq or (alpha_ratio < 0.5)

In [30]:
# Combine the previously flagged rows’ indexes
already_flagged_indexes = illegible_rows.index.union(new_illegible_rows.index)

# Keep only rows that haven't been flagged yet
strong_illegible_rows = strong_illegible_rows.loc[
    ~strong_illegible_rows.index.isin(already_flagged_indexes)
]

# Check results
strong_illegible_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text


In [27]:
new_illegible_rows.to_csv('illegible_rows_3.csv', index=False)

In [37]:
df[df['file_name'].str.contains('4-2-20 Save CA Ethnic Studies 1.pdf')]['text']

6988    Recipient:\n\nCalifornia Department of Educati...
Name: text, dtype: object

# Antisemitism Sampling

In [9]:
antisemitism_variants = [
    'antisemitism',
    'anti-semitism',
    'anti semitism',
    'antisemitic',
    'anti-semitic',
    'anti semitic',
    'antisemite',
    'anti-semite',
    'anti semite',
    'antisemitisim',
    'antisemitsm',
    'antisemitizm',
    'anti-semitizm',
    'anti semitizm',
    'ant1semitism',
    'anti$emitism',
    'anti-semit1sm',
    'anti semit1sm'
]

pattern = '|'.join(antisemitism_variants)

import pandas as pd

matched_rows = df[df['text'].str.contains(pattern, case=False, na=False)]

matched_rows

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
5,5,11-17-20 Yardeni.pdf,"Comments Received After September 30, 2020",1,0.000008,----------------------------------------------...
8,8,11-16-20 Landau.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: nonielandau Sent: Friday, November 13, 2..."
9,9,11-18-20 Donsky.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Joanne Donsky Sent: Wednesday, November ..."
...,...,...,...,...,...,...
7707,7708,10-12-20 Haufrect.pdf,"Comments Received After September 30, 2020",1,0.000008,Public Input Template–2020 Ethnic Studies Mode...
7708,7709,10-30-20 Meyers and Shmueli 2.pdf,"Comments Received After September 30, 2020",1,0.000008,Educators for Excellence in Ethnic Studies\n\n...
7709,7710,10-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 2, 20..."
7711,7712,10-14-20 George.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: IQC\nSent: Wednesday, October 14, 2020 1..."


In [13]:
antisemitism_samples = df.sample(n=40, replace=True, random_state=42)
antisemitism_samples

,Unnamed: 0,file_name,folder_name,submissions,weight,text
7270,7271,Williams.pdf,Comments Received After Field Review,1,0.000017,"From: Matthew Williams\nSent: Saturday, August..."
7603,7604,7-14-20 Hollingsworth.pdf,Comments Received After Field Review,1,0.000017,"From: Eliza Hollingsworth\nSent: Monday, July ..."
860,860,12-22-20 Stalnaker.pdf,Third Field Review (Dec 2020 - Jan 2021),1,0.000009,"From: Cecil Stalnaker Sent: Tuesday, December ..."
5390,5390,8-9-19 Shalev.pdf,First Field Review (June - August 2019),1,0.000018,"From: Maya Sent: Thursday, August 8, 2019 7:05..."
5226,5226,8-12-19 Codman.pdf,First Field Review (June - August 2019),1,0.000018,Public Input Template�2020 Ethnic Studies Mode...
5191,5191,8-15-19 Silver.pdf,First Field Review (June - August 2019),1,0.000018,"From: Sandy Silver Sent: Thursday, August 15, ..."
3772,3772,8-7-19 Harris 3.pdf,First Field Review (June - August 2019),1,0.000018,Public Input Template�2020 Ethnic Studies Mode...
3092,3092,8-7-19 Akselrud.pdf,First Field Review (June - August 2019),1,0.000018,"From: Marina Akselrud Sent: Wednesday, August ..."
5734,5734,8-9-19 Dickinson.pdf,First Field Review (June - August 2019),1,0.000018,"From Sent: Friday, August 9, 2019 5:35 PM To: ..."
6265,6266,9-30-20 sicklick .pdf,Second Field Review (Sept 2020),1,0.000011,"From: danielle sicklick Sent: Wednesday, Septe..."


# Recurring Authors

In [40]:
df = pd.read_csv('all_files.csv')
df

,Unnamed: 0,file_name,folder_name,submissions,weight,text
0,0,11-13-20 Khan.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Hareem Khan Sent: Thursday, November 12,..."
1,1,11-13-20 Solkovits.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: gs <gsolk@aol.com> Sent: Thursday, Novem..."
2,2,11-13-20 Kaur and Singh.pdf,"Comments Received After September 30, 2020",1,0.000008,"November 13th, 2020\n\nDear Superintendent Thu..."
3,3,11-19-20 Lamont.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sami Lamont Sent: Wednesday, November 18..."
4,4,11-18-20 Jensen.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Sine Hwang Jensen Sent: Tuesday, Novembe..."
...,...,...,...,...,...,...
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7714,7715,10-22-20 Nash.pdf,"Comments Received After September 30, 2020",1,0.000008,"Amin Nash, M.A. 10.21.2020\n\n1\n\nTo Executiv..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [60]:
authors = df['file_name'].str.extract(r'\d{1,2}-\d{1,2}-\d{2}\s+(.*?)\.pdf')

author_counts = authors[0].value_counts().reset_index()
author_counts.columns = ['author', 'count']

In [61]:
author_counts[author_counts['count'] > 1]

,author,count
0,Parker,46
1,Cohen,23
2,Smith,17
3,Schwartz,15
4,Levin,14
...,...,...
967,Herman,2
968,Bender,2
969,LeMay,2
970,Ellis,2


In [82]:
author_counts.head(10)

,author,count
0,Parker,46
1,Cohen,23
2,Smith,17
3,Schwartz,15
4,Levin,14
5,Levy,14
6,Klein,13
7,Lee,13
8,Gold,13
9,Parker 2,13


In [48]:
author_parker = df[df['file_name'].str.contains('Parker', case=False)]

In [51]:
author_parker[author_parker['text'].str.contains('Ruth', case=False)]
ruth_parker = author_parker.drop(['Unnamed: 0', 'submissions', 'weight'], axis=1)
ruth_parker

,file_name,folder_name,text
46,11-13-20 Parker Ella.pdf,"Comments Received After September 30, 2020","From: Ella Parker Sent: Friday, November 13, 2..."
54,11-12-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ella Parker Sent: Thursday, November 12,..."
61,11-29-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker Sent: Sunday, November 29, 2..."
75,11-13-20 Parker.pdf,"Comments Received After September 30, 2020","From: Maria Parker Sent: Friday, November 13, ..."
83,11-19-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker Sent: Thursday, November 19,..."
...,...,...,...
7706,10-12-20 Parker 2.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Monday, October 12, 2..."
7709,10-2-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Friday, October 2, 20..."
7712,10-27-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,10-30-20 Parker.pdf,"Comments Received After September 30, 2020","From: Ruth Parker\nSent: Friday, October 30, 2..."


In [52]:
ruth_parker.to_csv('ruth_parker.csv', index=False)

In [54]:
df[df['text'].str.contains('Ruth Parker', case=False)]

,Unnamed: 0,file_name,folder_name,submissions,weight,text
61,61,11-29-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Sunday, November 29, 2..."
83,83,11-19-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Thursday, November 19,..."
90,90,12-3-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Thursday, December 3, ..."
110,110,11-17-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Tuesday, November 17, ..."
139,139,11-10-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker Sent: Tuesday, November 10, ..."
...,...,...,...,...,...,...
7709,7710,10-2-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 2, 20..."
7712,7713,10-27-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Monday, October 26, 2..."
7713,7714,10-30-20 Parker.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Friday, October 30, 2..."
7715,7716,10-27-20 Parker 2.pdf,"Comments Received After September 30, 2020",1,0.000008,"From: Ruth Parker\nSent: Tuesday, October 27, ..."


In [58]:
print(df.iloc[7716]['text'])

From: Sue Fishkoff
Sent: Monday, October 26, 2020 6:28 PM
To: Ruth Parker
Cc: [list of other recipients redacted]; Ethnic Studies
Subject: [EXTERNAL] Re: Fw: A forward from J. Barry's friend, Marilyn; how can any self-respecting
person trust the NYT after knowing this?
Ruth,
This is Sue Fishkoff here, editor of J. The Jewish News of Northern California. I'm glad I'm on your email
list, because I can allay your fears.
What you have sent us is a piece of satire created in 2006, to protest the Israeli operation in Lebanon.
The real May 10, 1943 front page of the New York Times is here (you have to scroll down past the fake
front page, the explanation, and then you get to the real front page, which you can verify on the New
York Times website itself):
https://www.snopes.com/fact-check/warsaw-ghetto-uprising/
It is awful to see this true "fake news." Whatever you think of the New York Times, it would never have
published such Nazi propaganda.
Regards,
Sue
On Mon, Oct 26, 2020 at 5:41 PM Rut

In [64]:
author_counts[author_counts['author'].str.contains('parker', case=False)]

,author,count
0,Parker,46
9,Parker 2,13
97,Parker 3,5
2067,Parker 4,1
2318,Parker Rod,1
3926,Parker Ella,1
3985,Parker Ruth 2,1
3991,Parker Ruth,1


In [67]:
m = df[(df['text'].str.contains('Ruth Parker', case=False)) & (~df['file_name'].str.contains('Parker', case=False))]

In [78]:
from_ruth_parker = df[(df['text'].str.contains('Ruth Parker', case=False)) & (df['file_name'].str.contains('Parker', case=False))]

In [79]:
from_ruth_parker.to_csv('from_ruth_parker.csv', index=False)

In [80]:
m.to_csv('mentions_ruth_parker.csv', index=False)

Cohen

In [98]:
cohen = df[df['file_name'].str.contains('cohen', case=False)]

In [99]:
cohen['name'] = cohen['text'].str.extract(r'From:\s*(.*?)\s+Sent:', expand=False)
cohen['name'].value_counts()

/var/folders/p9/vxqrcpp154bd6rgfzqmcdgv40000gn/T/ipykernel_47703/2222924257.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cohen['name'] = cohen['text'].str.extract(r'From:\s*(.*?)\s+Sent:', expand=False)


name
Greg Cohen                     2
Eden Cohen                     2
Hilda Cohen                    2
Maayan Cohen                   1
gm cohen                       1
jonathan cohen                 1
Ross Glinkenhouse              1
Marc Cohen [email redacted]    1
Sharon hensel-cohen            1
Ross Cohen                     1
tav cohen                      1
David L Cohen                  1
Bruce Cohen                    1
Rachelle Cohen                 1
Farinaz Cohen                  1
Gail Cohen                     1
hcohenmb                       1
Liora Cohen                    1
Sheila Cohen                   1
Israel Cohen                   1
COHEN                          1
maayan cohen                   1
Jay Cohen                      1
Eli Cohen                      1
Lara Cohen                     1
Nazanin Lahijani Cohen         1
ARTHUR COHEN                   1
susan cohen                    1
Michael Cohen                  1
Marc Cohen                     1
Howar

Levin

In [96]:
levin = df[df['file_name'].str.contains('Levin', case=False)]

In [97]:
levin['name'] = levin['text'].str.extract(r'From:\s*(.*?)\s+Sent:', expand=False)
levin['name'].value_counts()

/var/folders/p9/vxqrcpp154bd6rgfzqmcdgv40000gn/T/ipykernel_47703/1124300334.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  levin['name'] = levin['text'].str.extract(r'From:\s*(.*?)\s+Sent:', expand=False)


name
Sarah Levin                                 4
Abby Levin                                  3
Kayla Levine                                1
David Levin                                 1
HEIDI LEVIN                                 1
Audrey Levine                               1
Alicia Levine                               1
Stephen Levinson                            1
Rachelli Levine                             1
Desiree levine                              1
Wendy Levine [mailto:wrlevine@gmail.com]    1
Steve Levin On Behalf Of Steve Levin        1
Susan Levine                                1
Jules Levin                                 1
Judith Levine                               1
Daniel Levinsohn                            1
Marc Levine                                 1
Boris Levine                                1
Edmund Levin                                1
Andrew Levine                               1
Rabbi Menachem Levine                       1
Fred levin                   